# Dynamic Memory

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Allocate memory whose size is known only while the program runs, with malloc and calloc, check for NULL, and free it.
- Grow a buffer with realloc, doubling its capacity, and explain why the old pointer must not be used afterwards.
- Build arrays of rows with different lengths, or one block indexed by row and column.

**In this notebook**, one idea at a time (run each cell before moving on):
1. Why ask for memory while running?
2. malloc, sizeof and free
3. calloc: a block that starts at zero
4. realloc: growing a block
5. Memory that outlives the function
6. The three memory bugs

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check_int and check_str print ✓ or ✗; the exercises below use them
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected %ld, got %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected \"%s\", got \"%s\"\n", what, expected, actual);
    }
}

## 1. Why ask for memory while running?

An array's size is fixed when you write the code, and a local array disappears when its function returns. Both are a problem when the size comes from the data (how many numbers will the user type?) or when the data must outlive the function. The answer is the **heap**: a big area where the program asks for blocks while it runs, and gives them back when it's done.

| | Stack (local variables) | Heap (`malloc`) |
|---|---|---|
| size | fixed in the code | chosen while running |
| lasts | until the function returns | until you `free` it |
| room | a few megabytes | most of the computer's memory |

In [ ]:
int n = 5;                              // imagine this came from the input
int *squares = malloc(n * sizeof *squares);
if (squares == NULL) {
    return 1;                           // no memory: give up
}
for (int i = 0; i < n; i++) {
    squares[i] = i * i;                 // used just like an array
}
for (int i = 0; i < n; i++) {
    printf("%d ", squares[i]);
}
printf("\n");
free(squares);                          // give it back

🧪 **Try it:** change `n` to 12 and run: the size is just a variable now.

## 2. malloc, sizeof and free

`malloc(bytes)` reserves a block of that many bytes and returns a pointer to it, or `NULL` when there's no memory. Compute the bytes with `sizeof`: `n * sizeof *p` is "n elements of whatever `p` points to", which stays right even if you change `p`'s type. The block's contents start as **garbage**. When you're done, `free(p)` gives it back. The slides follow a block from `malloc` to `free`.

#### 🎞️ malloc, realloc and free

![malloc reserves a block in the heap, realloc grows it (maybe moving it), and free gives it back while the pointer keeps the old address until it is set to NULL](malloc-steps.svg)

In [ ]:
double *prices = malloc(3 * sizeof *prices);
if (prices == NULL) {
    return 1;
}
printf("asked for %zu bytes\n", 3 * sizeof *prices);
prices[0] = 2.5;
prices[1] = 4.0;
prices[2] = 1.25;
double total = 0;
for (int i = 0; i < 3; i++) {
    total += prices[i];
}
printf("total %.2f\n", total);
free(prices);
prices = NULL;          // so nobody uses the old address by mistake

🧪 **Try it:** read `prices[0]` after the `free` (and before `prices = NULL`) and run: the memory check reports a "heap-use-after-free".

## 3. calloc: a block that starts at zero

`calloc(count, size)` is like `malloc(count * size)`, but the block starts **filled with zeros**. Use it for counters and tables that start empty.

In [ ]:
const char *text = "mississippi";
int *counts = calloc(26, sizeof *counts);
if (counts == NULL) {
    return 1;
}
for (int i = 0; text[i] != '\0'; i++) {
    counts[text[i] - 'a']++;
}
for (int c = 0; c < 26; c++) {
    if (counts[c] > 0) {
        printf("%c: %d\n", 'a' + c, counts[c]);
    }
}
free(counts);

🧪 **Try it:** count the letters of another word. What would happen with a capital letter or a space? (Its index would fall outside 0 to 25.)

### ✏️ Exercise: A range in the heap

Return a new block of n ints holding 0, 1, …, n − 1. The caller frees it.

In [ ]:
int *range_array(int n) {
    // TODO: malloc n ints and fill them
    return calloc(n, sizeof(int));
}

int main(void) {
    int *r = range_array(4);
    check_int("r[0]", r[0], 0);
    check_int("r[3]", r[3], 3);
    free(r);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
int *range_array(int n) {
    int *a = malloc(n * sizeof(int));
    if (a == NULL) {
        return NULL;
    }
    for (int i = 0; i < n; i++) {
        a[i] = i;
    }
    return a;
}

int main(void) {
    int *r = range_array(4);
    check_int("r[0]", r[0], 0);
    check_int("r[3]", r[3], 3);
    free(r);
    return 0;
}
```

</details>

## 4. realloc: growing a block

When a block fills up, `realloc(p, new_bytes)` gives you a bigger one, keeping the old values. It **may move** the block to another address, so always use the pointer it returns. It returns `NULL` on failure, leaving the old block intact, so store the result in a new variable first. A common strategy is to **double** the capacity each time it fills, so growing stays fast.

In [ ]:
int capacity = 2;
int count = 0;
int *values = malloc(capacity * sizeof *values);
if (values == NULL) {
    return 1;
}
for (int v = 10; v <= 70; v += 10) {
    if (count == capacity) {
        capacity *= 2;
        int *bigger = realloc(values, capacity * sizeof *values);
        if (bigger == NULL) {
            free(values);
            return 1;
        }
        values = bigger;
        printf("grew to %d\n", capacity);
    }
    values[count++] = v;
}
for (int i = 0; i < count; i++) {
    printf("%d ", values[i]);
}
printf("\n");
free(values);

🧪 **Try it:** change the loop to go up to 1000: how many times does it grow? Then grow by 1 instead of doubling and count the `realloc` calls.

## 5. Memory that outlives the function

A heap block lasts until it's freed, even after the function that made it returns. So a function can build something on the heap and **return the pointer**. In return, the caller becomes responsible for calling `free`. Write that rule in a comment above the function.

In [ ]:
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

/* Returns "word word word…" (times copies) in a new block. The caller frees it. */
char *repeat(const char *word, int times) {
    size_t len = strlen(word);
    char *out = malloc(times * (len + 1) + 1);
    if (out == NULL) {
        return NULL;
    }
    out[0] = '\0';
    for (int i = 0; i < times; i++) {
        if (i > 0) {
            strcat(out, " ");
        }
        strcat(out, word);
    }
    return out;
}

int main(void) {
    char *cheer = repeat("go", 3);
    if (cheer == NULL) {
        return 1;
    }
    printf("%s!\n", cheer);
    free(cheer);
    return 0;
}

🧪 **Try it:** change `repeat` to use a local array `char out[100];` and return it: read the warning. That array disappears when the function returns.

### ✏️ Exercise: Duplicate a string

Return a copy of s in a new heap block. Remember the extra byte for `'\0'`.

In [ ]:
char *duplicate(const char *s) {
    // TODO: malloc strlen(s) + 1 bytes, then strcpy
    return calloc(1, 1);
}

int main(void) {
    char *d = duplicate("hello");
    check_str("the copy", d, "hello");
    char *e = duplicate("");
    check_str("an empty copy", e, "");
    free(d);
    free(e);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
char *duplicate(const char *s) {
    char *copy = malloc(strlen(s) + 1);
    if (copy == NULL) {
        return NULL;
    }
    strcpy(copy, s);
    return copy;
}

int main(void) {
    char *d = duplicate("hello");
    check_str("the copy", d, "hello");
    char *e = duplicate("");
    check_str("an empty copy", e, "");
    free(d);
    free(e);
    return 0;
}
```

</details>

## 6. The three memory bugs

- **Use after free:** after `free(p)`, `p` still holds the old address, but the block is no longer yours. Set `p = NULL` right after freeing.
- **Double free:** freeing the same block twice corrupts the heap.
- **Leak:** losing the last pointer to a block without freeing it. The memory stays taken until the program ends; in a long-running program, leaks add up.

Sphinx's memory checks stop the program at the first two and show the lines involved.

In [ ]:
int *a = malloc(sizeof *a);
if (a == NULL) {
    return 1;
}
*a = 42;
printf("%d\n", *a);
free(a);
a = NULL;
if (a != NULL) {
    printf("never reached\n");
}

🧪 **Try it:** delete `a = NULL;` and add a second `free(a);`: read the "double-free" report. (`free(NULL)` does nothing, which is another reason to set pointers to `NULL`.)

## In short

Arrays have a size fixed when you write the code. When the size comes from the data, ask for memory while the program runs: `malloc(bytes)` returns a pointer to a new block on the **heap**, or `NULL` when there is none. The block lasts until you `free` it, even after the function that made it returns.

```c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    int capacity = 4, count = 0, value;
    int *values = malloc(capacity * sizeof *values);
    if (values == NULL) return 1;
    while (scanf("%d", &value) == 1) {
        if (count == capacity) {                 // full: double it
            capacity *= 2;
            int *bigger = realloc(values, capacity * sizeof *values);
            if (bigger == NULL) { free(values); return 1; }
            values = bigger;
        }
        values[count++] = value;
    }
    printf("%d numbers\n", count);
    free(values);
    return 0;
}
```

`realloc` may move the block, so keep using the pointer it returns.

**Watch out:** after `free(p)`, `p` still holds the old address. Using it, or freeing it twice, is undefined behavior; memory checks catch it.

## Readings

- [Manual Memory Allocation](https://beej.us/guide/bgc/html/split/manual-memory-allocation.html) (Beej's Guide to C Programming, chapter 12, 12 min): malloc and free, error checking, calloc, and realloc, including "Reading in Lines of Arbitrary Length", which is the Longest Line challenge almost line by line.
- [Lecture 4: Copying and malloc](https://cs50.harvard.edu/x/2026/notes/4/#copying-and-malloc) (CS50x (Harvard), lecture notes, 10 min): Optional: from "Copying and malloc" to "Garbage Values": why copying a pointer doesn't copy the string, and the stack and the heap in a picture.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Unknown Count** · Easy
- **Longest Line** · Medium
- **Pascal's Triangle** · Medium
- **Matrix Multiply** · Hard